# Exercice 1

In [ ]:
%pip install -r requirements.txt

import pandas as pd
import numpy as np
from great_tables import GT

df = pd.read_csv(
 'https://www.data.gouv.fr/fr/datasets/r/182268fc-2103-4bcb-a850-6cf90b02a9eb'
)


### Question 1

In [ ]:
#Verications préliminaires
print(df.dtypes)

In [3]:
# Modification de la variable code_commune
df['code_commune'] = (
    df['code_departement'].astype(str).str.zfill(2) +
    df['code_commune'].astype(str).str.zfill(3)
    )

# Création de la variable candidat
df['candidat'] = (
    df['prenom'].fillna('') + 
    ' ' + 
    df['nom'].fillna('')
).str.strip()

### Question 2 

In [ ]:
# Connaitre les valeurs que prend candidat pour enlever les valeurs qui n'en sont pas 
print(df['candidat'].unique())

In [ ]:
#Exclusion des votes non exprimés (abstentions, blancs, nuls)
votes_non_exprimes = ['abstentions', 'blancs', 'nuls']

# Filtrage pour ne garder que les candidats
candidats_df = df[~df['candidat'].isin(votes_non_exprimes)]

# Comptage du nombre de candidats uniques
candidats = candidats_df['candidat'].nunique()

#Affichage
print(f"En 2022, il y avait {candidats} candidats à l'élection présidentielle.")


### Question 3

In [ ]:
# Fitrage 
votes_non_exprimes = ['abstentions', 'blancs', 'nuls']
candidats_df = df[~df['candidat'].isin(votes_non_exprimes)]

#Calculs des valeurs
scores_nationaux = candidats_df.groupby('candidat', as_index=False)['voix'].sum()

scores_nationaux = scores_nationaux.sort_values('voix', ascending=False)

total_voix = scores_nationaux['voix'].sum()

scores_nationaux['pourcentage'] = scores_nationaux['voix'] / total_voix

# Construction du beau tableau 
gt_scores = (
    GT(scores_nationaux)
    .tab_header(
        title="Elections",
        subtitle="Résultats du premier tour (10 avril 2022)"
    )
    .cols_label(
        {
            "candidat": "Candidat",
            "voix": "Nombre votes (total)",
            "pourcentage": "Score (% votes exprimés)" 
        }
    )
    .fmt_integer(columns="voix", sep_mark=" ")
    # Formatage en pourcentage (0.25 devient 25.00%)
    .fmt_percent(columns="pourcentage", decimals=2)
)

gt_scores